# Cloud Provider Analytics · Notebook 03 — MVP final (Landing → Bronze → Silver → Gold → AstraDB)

**Big Data · ITBA · 2C 2026 · Entrega final (MVP integrado)**
Grupo: Cayetana Peces · Abril Solé · Agustina Edalian · Delfina Vázquez Escati

Es el pipeline completo de la entrega final: el notebook 02 (2ª entrega) queda como antecedente. El notebook orquesta el pipeline; la lógica está en `src/` (funciones chicas y testeadas en `tests/`) y la configuración en `config/pipeline.example.yaml`.

| Sección | Qué hace | Responsable |
|---|---|---|
| 0. Setup | Repo, configuración, SparkSession | Delfina |
| 1. Landing | Descompresión e inventario (solo lectura) | Cayetana |
| 2. Bronze batch | 7 maestros con esquema explícito, quarantine y dedupe | Cayetana |
| 3. Bronze streaming | Structured Streaming: watermark, dedupe por `event_id`, late data, checkpoint | Cayetana |
| 4. Silver | `dim_org`, eventos, facturación en USD y tickets; reglas de calidad y quarantine | Abril |
| 5. Analítica | Anomalías de costo con MAD por (servicio, métrica) | Abril |
| 6. Gold | Los 5 marts (FinOps, Soporte, Producto/GenAI) + vistas de serving | Agustina |
| 7. Idempotencia | Re-ejecución sin duplicados (conteos antes/después) | Delfina |
| 8. Serving | Carga a AstraDB y las 5 consultas (Q1–Q5) | Agustina |
| 9. Evidencias | run_log, quarantine, rutas, tamaños | Delfina |

**Para correrlo en Colab:** ver el Quickstart del README (subir el dataset y el bundle de AstraDB, cargar el token en *Secrets* y *Ejecutar todo*).

## 0. Setup

In [1]:
import os, sys

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/delfivazqueze/cloud-provider-analytics.git"

if EN_COLAB:
    !pip -q install pyspark==3.5.3 cassandra-driver pyyaml setuptools
    if not os.path.exists("/content/cloud-provider-analytics"):
        !git clone -q {REPO_URL} /content/cloud-provider-analytics
    REPO = "/content/cloud-provider-analytics"
else:
    REPO = os.path.abspath("..")          # corriendo desde notebooks/ del repo
sys.path.insert(0, REPO)

import pyspark
print("PySpark", pyspark.__version__)

PySpark 3.5.3


In [2]:
import glob, json, time, zipfile, shutil
from pyspark.sql import SparkSession, functions as F

from src.common.config import load_config
from src.common.run_log import RunLog, new_run_id

cfg = load_config(REPO, base="/content")
P = cfg["paths"]

spark = (SparkSession.builder
         .appName(cfg["spark"]["app_name"])
         .config("spark.sql.shuffle.partitions", cfg["spark"]["shuffle_partitions"])
         .config("spark.sql.sources.partitionOverwriteMode", cfg["spark"]["partition_overwrite_mode"])
         .config("spark.sql.session.timeZone", cfg["spark"]["timezone"])
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

RUN_ID = new_run_id()
log = RunLog(spark, P["meta"], RUN_ID)
os.makedirs(P["evidence"], exist_ok=True)
print("Spark", spark.version, "| run_id:", RUN_ID)
print(json.dumps({k: v for k, v in cfg.items() if k != "paths"}, indent=1))

Spark 3.5.3 | run_id: 20261118T130005-cadbac
{
 "spark": {
  "app_name": "cloud-provider-analytics",
  "shuffle_partitions": 8,
  "partition_overwrite_mode": "dynamic",
  "timezone": "UTC"
 },
 "streaming": {
  "max_files_per_trigger": 30,
  "watermark": "60 days",
  "trigger": "available_now",
  "processing_time": "5 minutes"
 },
 "quality": {
  "anomaly_method": "mad_by_service_metric",
  "anomaly_threshold": 3.5,
  "nps_min": -100,
  "nps_max": 100,
  "csat_min": 1,
  "csat_max": 5,
  "currencies": [
   "USD",
   "ARS",
   "EUR"
  ]
 },
 "cassandra": {
  "keyspace": "cloud_analytics",
  "secure_bundle_env": "ASTRA_SCB_PATH",
  "token_env": "ASTRA_DB_APPLICATION_TOKEN",
  "concurrency": 50
 }
}


## 1. Landing

Landing es inmutable: se descomprime una sola vez y después solo se lee. Para empezar de cero (demo) alcanza con borrar `datalake/` salvo `landing/` (celda de limpieza al final).

In [3]:
zip_path = f"{P['base']}/cloud_provider_challenge_dataset_v1.zip"
if not os.path.exists(f"{P['landing']}/customers_orgs.csv"):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(P["base"])
archivos_eventos = sorted(glob.glob(f"{P['landing']}/usage_events_stream/*.jsonl"))
print("CSV en landing:", sorted(os.path.basename(f) for f in glob.glob(f"{P['landing']}/*.csv")))
print("Archivos de eventos:", len(archivos_eventos))

CSV en landing: ['billing_monthly.csv', 'customers_orgs.csv', 'marketing_touches.csv', 'nps_surveys.csv', 'resources.csv', 'support_tickets.csv', 'users.csv']
Archivos de eventos: 120


## 2. Bronze batch: maestros y facturación

Se ingestan los 7 CSV (la consigna pide al menos 3). Para cada uno:
- esquema explícito (`src/common/schemas.py`) y modo `PERMISSIVE` con columna `_corrupt_record`: una fila mal formada no rompe la carga, va a quarantine (`B01`);
- clave natural nula → quarantine (`B02`); duplicados por clave natural → se descartan;
- columnas técnicas `ingest_ts`, `ingest_date`, `source_file`, `run_id`;
- partición por `ingest_date` (snapshot del día) y `billing_month` para facturación. El overwrite dinámico reemplaza solo la partición del día, así que re-ejecutar no duplica.

In [4]:
from src.common.schemas import CSV_SCHEMAS
from src.jobs.bronze_batch import ingest_master

print(f"{'maestro':<20}{'entrada':>9}{'bronze':>9}{'quarantine':>12}")
for name in CSV_SCHEMAS:
    with log.track("bronze_batch", "bronze", name):        # si falla, queda registrado como ERROR en run_log
        n_in, n_ok, n_q = ingest_master(spark, cfg, name, RUN_ID, log)
    print(f"{name:<20}{n_in:>9,}{n_ok:>9,}{n_q:>12,}")
log.flush()

maestro               entrada   bronze  quarantine


customers_orgs             80       80           0


users                     800      800           0


resources                 400      400           0


support_tickets         1,000    1,000           0


marketing_touches       1,500    1,500           0


nps_surveys                92       92           0


billing_monthly           240      240           0


> **Hallazgo de la regla B01.** En la primera corrida la regla mandó 211 filas de `resources` a quarantine como corruptas. El motivo: `tags_json` trae comillas escapadas al estilo CSV estándar (`"[""env:prod""]"`) y Spark, por defecto, escapa con `\`. Se agregó `escape='"'` a la lectura y desde entonces hay 0 filas corruptas. En la E1 (sin columna `_corrupt_record`) esas filas se leían con el campo truncado sin que nadie se enterara.

## 3. Bronze streaming: `usage_events_stream/*.jsonl`

### 3.1 Elegir el watermark con datos

El watermark es `máx(event_time visto) − tolerancia`, y todo evento más viejo que eso se descarta por tardío. En la E1 medimos que **cada archivo trae eventos de los 60 días** (`evidence/01_event_time_range_by_file.csv`). Antes de fijar el valor, corremos la misma lectura + dedupe con dos tolerancias contra un sink `noop` (no escribe nada) y medimos `numRowsDroppedByWatermark`.

In [5]:
from src.jobs.bronze_stream import watermark_experiment

experimento = []
for wm in ["2 days", "60 days"]:
    r = watermark_experiment(spark, cfg, wm)
    r["watermark"] = wm
    experimento.append(r)
    print(r)

{'micro_lotes': 4, 'filas_leidas': 43200, 'descartadas_por_watermark': 20874, 'emitidas': 22326, 'claves_en_estado': 2181, 'watermark': '2 days'}


{'micro_lotes': 4, 'filas_leidas': 43200, 'descartadas_por_watermark': 0, 'emitidas': 43200, 'claves_en_estado': 43200, 'watermark': '60 days'}


Con 2 días, a partir del segundo micro-lote el watermark queda cerca del 29/08 y **se pierde casi la mitad de los eventos** (todo lo que llega en los lotes 2 a 4 con fecha anterior). Con **60 días** no se pierde ningún evento y el estado del dedupe son ~43 mil `event_id` (chico). Se fija `watermark = "60 days"` (D15). En un caso real con eventos ordenados se usaría una tolerancia corta y el recálculo batch de Lambda corrige lo que llegue más tarde.

### 3.2 Ingesta en streaming

- `readStream` con esquema explícito y `maxFilesPerTrigger = 30` (admission control): 120 archivos → 4 micro-lotes.
- Reglas de Bronze: `event_id` no nulo (`B02`), timestamp parseable (`B03`), `schema_version ∈ {1,2}` (`B04`). Los inválidos van por una **segunda query** a quarantine.
- `withWatermark("event_ts", "60 days")` + `dropDuplicatesWithinWatermark(["event_id"])`.
- Sink Parquet particionado por `event_date`, `outputMode("append")` y **checkpoint propio por query**.
- `trigger(availableNow=True)`: procesa todo lo pendiente en micro-lotes y se detiene (en producción se cambia a `processingTime="5 minutes"` desde la config, mismo código).

In [6]:
from src.jobs.bronze_stream import start_bronze_queries, progress_summary

t0 = time.time()
q_bronze, q_quar = start_bronze_queries(spark, cfg, RUN_ID)
q_bronze.awaitTermination(); q_quar.awaitTermination()
dur = time.time() - t0

prog_bronze = progress_summary(q_bronze)
print("bronze_usage_events :", prog_bronze)

def hay_parquet(path):
    return len(glob.glob(f"{path}/**/*.parquet", recursive=True)) > 0

bronze_ev = spark.read.parquet(f"{P['bronze']}/usage_events")
n_bronze = bronze_ev.count()
n_q_bronze = (spark.read.parquet(f"{P['quarantine']}/usage_events_bronze").count()
              if hay_parquet(f"{P['quarantine']}/usage_events_bronze") else 0)
print(f"Bronze usage_events: {n_bronze:,} filas | quarantine Bronze: {n_q_bronze:,}")
log.add("bronze_stream", "bronze", "usage_events", prog_bronze["filas_leidas"], n_bronze, n_q_bronze,
        prog_bronze["filas_leidas"] - n_q_bronze - n_bronze, dur,
        detalle=f"micro_lotes={prog_bronze['micro_lotes']}; tarde={prog_bronze['descartadas_por_watermark']}")
log.flush()

bronze_usage_events : {'micro_lotes': 4, 'filas_leidas': 43200, 'descartadas_por_watermark': 0, 'emitidas': 43200, 'claves_en_estado': 43200}


Bronze usage_events: 43,200 filas | quarantine Bronze: 0


In [7]:
# Métricas del último micro-lote (lo que se miraría en producción para alertar)
lp = q_bronze.lastProgress
print(json.dumps({"batchId": lp["batchId"], "numInputRows": lp["numInputRows"],
                  "durationMs": lp["durationMs"], "stateOperators": lp["stateOperators"],
                  "sources": [{k: s[k] for k in ("description", "startOffset", "endOffset")} for s in lp["sources"]]},
                 indent=1, default=str)[:2500])
print("\nCheckpoint:", sorted(os.listdir(f"{P['checkpoints']}/bronze_usage_events")))

{
 "batchId": 4,
 "numInputRows": 0,
 "durationMs": {
  "addBatch": 671,
  "commitOffsets": 52,
  "getBatch": 0,
  "latestOffset": 0,
  "queryPlanning": 19,
  "triggerExecution": 776,
  "walCommit": 31
 },
 "stateOperators": [
  {
   "operatorName": "dedupeWithinWatermark",
   "numRowsTotal": 43200,
   "numRowsUpdated": 0,
   "allUpdatesTimeMs": 31,
   "numRowsRemoved": 0,
   "allRemovalsTimeMs": 5,
   "commitTimeMs": 443,
   "memoryUsedBytes": 11321728,
   "numRowsDroppedByWatermark": 0,
   "numShufflePartitions": 8,
   "numStateStoreInstances": 8,
   "customMetrics": {
    "loadedMapCacheHitCount": 32,
    "loadedMapCacheMissCount": 0,
    "numDroppedDuplicateRows": 0,
    "stateOnCurrentVersionSizeBytes": 7869504
   }
  }
 ],
 "sources": [
  {
   "description": "FileStreamSource[file:/content/datalake/landing/usage_events_stream]",
   "startOffset": {
    "logOffset": 3
   },
   "endOffset": {
    "logOffset": 3
   }
  }
 ]
}

Checkpoint: ['.metadata.crc', 'commits', 'metadata', 'of

## 4. Silver

### 4.1 `dim_org` (maestro conformado)
Último snapshot de Bronze (SCD tipo 1 en el MVP), tipos explícitos (`is_enterprise` booleano, `signup_date` fecha), strings normalizados y NPS fuera de [−100, 100] → `null` + flag `is_nps_out_of_range`.

In [8]:
from src.jobs.silver import build_dim_org, build_silver_events

with log.track("silver_dim_org", "silver", "dim_org"):
    dim_org = build_dim_org(spark, cfg, RUN_ID, log)
dim_org.printSchema()
dim_org.filter("is_nps_out_of_range").select("org_id", "org_name", "nps_score", "is_nps_out_of_range").show()

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)
 |-- is_nps_out_of_range: boolean (nullable = true)
 |-- source_file: string (nullable = true)
 |-- updated_ts: timestamp (nullable = true)
 |-- run_id: string (nullable = true)



+------------+--------------+---------+-------------------+
|      org_id|      org_name|nps_score|is_nps_out_of_range|
+------------+--------------+---------+-------------------+
|org_pac56t4u|Gamma Cloud 37|     NULL|               true|
+------------+--------------+---------+-------------------+



### 4.2 Eventos conformados

Pasos (`src/jobs/silver.py`):
1. Normalización de `service`, `region`, `metric`, `unit` (minúsculas, sin espacios).
2. **v1/v2 compatibles:** `carbon_kg` y `genai_tokens` quedan `null` en v1 (no 0: "no medido" ≠ "cero").
3. **Regla R3 — `unit` no nulo cuando hay `value`:** se imputa desde `metric` (relación 1:1 medida en la E1) y queda el flag `unit_imputed`. Si la métrica no es conocida → quarantine `S02`.
4. **Regla R2 — costos negativos (D29):** `cost_usd_increment` es el cargo por un consumo medido; los créditos y ajustes comerciales se registran en facturación. Un costo negativo **con consumo** (`value > 0`) es un error de signo: se corrige al valor absoluto, queda el original en `cost_usd_raw` y el flag `is_cost_sign_corrected`. **Sin consumo** no se puede validar: quarantine `S01` con `is_cost_anomaly = true`. La evidencia está en la sección 4.2.1.
5. **Join de enriquecimiento** con `dim_org` (broadcast: 80 filas) y regla `S03` (org inexistente).
6. Features a nivel evento: `requests`, `cpu_hours`, `storage_gb_hours`, `genai_tokens`, `carbon_kg`.
7. Score de anomalía (sección 5).

In [9]:
with log.track("silver_usage_events", "silver", "usage_events"):
    silver_ev = build_silver_events(spark, cfg, dim_org, RUN_ID, log)
log.flush()

quar_silver = spark.read.parquet(f"{P['quarantine']}/usage_events_silver")
print(f"Bronze: {n_bronze:,} | Silver: {silver_ev.count():,} | Quarantine Silver: {quar_silver.count():,}")
print("Conciliación (O2) → Bronze == Silver + Quarantine:", n_bronze == silver_ev.count() + quar_silver.count())
quar_silver.groupBy("rule_id", "motivo").count().show(truncate=False)

Bronze: 43,200 | Silver: 43,196 | Quarantine Silver: 4


Conciliación (O2) → Bronze == Silver + Quarantine: True
+------------------------------+------------------------------------------------------------------------------------+-----+
|rule_id                       |motivo                                                                              |count|
+------------------------------+------------------------------------------------------------------------------------+-----+
|S01_COSTO_NEGATIVO_SIN_CONSUMO|cost_usd_increment < 0 sin consumo medido (value nulo): no se puede validar el signo|4    |
+------------------------------+------------------------------------------------------------------------------------+-----+



In [10]:
# Compatibilidad de esquema (O4): todos los v1 y v2 de Bronze están en Silver o explicados en quarantine
b = bronze_ev.groupBy("schema_version").count().withColumnRenamed("count", "bronze")
s = silver_ev.groupBy("schema_version").agg(F.count("*").alias("silver"),
        F.sum(F.col("carbon_kg").isNotNull().cast("int")).alias("con_carbon"),
        F.sum(F.col("genai_tokens").isNotNull().cast("int")).alias("con_tokens"))
qv = quar_silver.groupBy("schema_version").count().withColumnRenamed("count", "quarantine")
b.join(s, "schema_version").join(qv, "schema_version", "left").orderBy("schema_version").show()

# Regla R3: cuántas unit se imputaron y por métrica
silver_ev.filter("unit_imputed").groupBy("metric", "unit").count().orderBy("metric").show()
print("Eventos con value y sin unit que quedan en Silver:",
      silver_ev.filter(F.col("value_num").isNotNull() & F.col("unit").isNull()).count())

+--------------+------+------+----------+----------+----------+
|schema_version|bronze|silver|con_carbon|con_tokens|quarantine|
+--------------+------+------+----------+----------+----------+
|             1| 10800| 10799|         0|         0|         1|
|             2| 32400| 32397|     32397|      3132|         3|
+--------------+------+------+----------+----------+----------+



+----------------+--------+-----+
|          metric|    unit|count|
+----------------+--------+-----+
|       cpu_hours|   hours|  502|
|        requests|   count|  898|
|storage_gb_hours|gb_hours|  638|
+----------------+--------+-----+

Eventos con value y sin unit que quedan en Silver: 0


#### 4.2.1 Semántica de los costos negativos (D29)

Antes de decidir qué hacer con los costos negativos medimos qué son. Si fueran créditos o reversas, deberían anular un cargo anterior del mismo recurso o concentrarse en algunos clientes; si fueran errores de signo, deberían parecerse a un consumo normal.

In [11]:
neg = bronze_ev.filter("cost_usd_increment < 0")
pos = bronze_ev.filter("cost_usd_increment > 0")

# 1) ¿Hay consumo medido?
print(f"Eventos con costo negativo: {neg.count()} | con value > 0: {neg.filter('value_num > 0').count()} "
      f"| sin value: {neg.filter('value_num IS NULL').count()}")

# 2) ¿El monto es el precio normal de su (servicio, métrica) con el signo invertido?
precio = (pos.filter("value_num > 0").groupBy("service", "metric")
             .agg(F.percentile_approx(F.col("cost_usd_increment") / F.col("value_num"), 0.5).alias("precio_unitario")))
ratio = (neg.filter("value_num > 0").join(precio, ["service", "metric"])
            .select((F.abs("cost_usd_increment") / F.col("value_num") / F.col("precio_unitario")).alias("r")))
print("|costo| / consumo comparado con el precio unitario típico (p10, p50, p90):",
      [round(x, 2) for x in ratio.approxQuantile("r", [0.1, 0.5, 0.9], 0.001)])

# 3) ¿Anulan un cargo anterior? Cargo positivo del mismo recurso por el mismo monto
espejo = (neg.alias("n").join(pos.alias("p"), (F.col("n.resource_id") == F.col("p.resource_id")) &
          (F.abs(F.col("p.cost_usd_increment") + F.col("n.cost_usd_increment")) < 1e-4)).select("n.event_id").distinct().count())
azar = (pos.alias("a").join(pos.alias("b"), (F.col("a.resource_id") == F.col("b.resource_id")) &
        (F.col("a.event_id") < F.col("b.event_id")) &
        (F.abs(F.col("a.cost_usd_increment") - F.col("b.cost_usd_increment")) < 1e-4)).count())
print(f"Negativos con un cargo 'espejo' en el mismo recurso: {espejo} "
      f"(como referencia, {azar} pares de cargos positivos coinciden por azar)")

# 4) ¿Se concentran en algunos clientes?
print(f"Organizaciones con costos negativos: {neg.select('org_id').distinct().count()} de 80 | "
      f"servicios: {neg.select('service').distinct().count()} de 6")

# Resultado de la decisión en Silver
corr = silver_ev.filter("is_cost_sign_corrected")
print(f"\nCorregidos en Silver: {corr.count()} eventos | costo original {corr.agg(F.round(F.sum('cost_usd_raw'), 2)).first()[0]:,} USD "
      f"→ corregido {corr.agg(F.round(F.sum('cost_usd_increment'), 2)).first()[0]:,} USD")

Eventos con costo negativo: 216 | con value > 0: 212 | sin value: 4


|costo| / consumo comparado con el precio unitario típico (p10, p50, p90): [0.82, 1.02, 1.18]


Negativos con un cargo 'espejo' en el mismo recurso: 4 (como referencia, 380 pares de cargos positivos coinciden por azar)


Organizaciones con costos negativos: 68 de 80 | servicios: 6 de 6



Corregidos en Silver: 212 eventos | costo original -895.21 USD → corregido 895.21 USD


**Lectura.** Los costos negativos tienen consumo medido, su monto es el precio normal de su servicio con el signo invertido, no anulan cargos anteriores (las coincidencias son las mismas que se dan por azar) y están repartidos entre muchas organizaciones y todos los servicios. No se comportan como créditos (que el negocio registra en facturación: `credits` y notas de crédito) sino como errores de signo en la telemetría. Por eso se corrigen y quedan marcados, en lugar de mandarlos a quarantine (subestimaría el costo) o restarlos (lo subestimaría todavía más). El impacto en el costo total queda en `DECISIONS.md` (D29).

In [12]:
# Muestras de quarantine (evidencia verificable de las reglas)
(quar_silver.select("rule_id", "event_id", "event_ts", "org_id", "service", "metric", "cost_usd_increment",
                    "is_cost_anomaly", "motivo", "source_file")
    .orderBy("cost_usd_increment").show(5, truncate=40))

+------------------------------+----------------+-------------------+------------+--------+----------------+------------------+---------------+----------------------------------------+----------------------------------------+
|                       rule_id|        event_id|           event_ts|      org_id| service|          metric|cost_usd_increment|is_cost_anomaly|                                  motivo|                             source_file|
+------------------------------+----------------+-------------------+------------+--------+----------------+------------------+---------------+----------------------------------------+----------------------------------------+
|S01_COSTO_NEGATIVO_SIN_CONSUMO|evt_ojme129ymwaz|2025-08-30 17:06:00|org_he7ur23g| compute|        requests|           -10.563|           true|cost_usd_increment < 0 sin consumo me...|file:///content/datalake/landing/usag...|
|S01_COSTO_NEGATIVO_SIN_CONSUMO|evt_5onpviqd2o0w|2025-07-16 20:39:00|org_nam148p0| compute|     

In [13]:
# El join con dim_org es un broadcast hash join (sin shuffle de los eventos)
from src.jobs.silver import conform_events
conform_events(bronze_ev, dim_org).explain(mode="simple")

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [org_id#10409, event_id#10407, event_ts#10408-T5184000000ms, resource_id#10410, service#18488, region#18507, metric#18526, value_raw#10414, value_num#10415, unit#18584, cost_usd_increment#18647, schema_version#10418, carbon_kg#10419, genai_tokens#10420L, ingest_ts#10421, source_file#10422, run_id#10423, event_date#10424, unit_imputed#18564, cost_usd_raw#18604, is_cost_sign_corrected#18625, requests#18713, cpu_hours#18736, storage_gb_hours#18760, ... 4 more fields]
   +- BroadcastHashJoin [org_id#10409], [org_id#10626], LeftOuter, BuildRight, false
      :- Project [event_id#10407, event_ts#10408-T5184000000ms, org_id#10409, resource_id#10410, service#18488, region#18507, metric#18526, value_raw#10414, value_num#10415, unit#18584, CASE WHEN is_cost_sign_corrected#18625 THEN abs(cost_usd_raw#18604) ELSE cost_usd_raw#18604 END AS cost_usd_increment#18647, schema_version#10418, carbon_kg#10419, genai_tokens#10420L, ingest_t

### 4.3 Facturación en USD (`billing_usd`)

`revenue_usd = (subtotal − credits + taxes) × exchange_rate_to_usd` (D12): el tipo de cambio de ARS (≈0,0015) confirma que es "USD por unidad de moneda". `credits` nulo = 0. Subtotal negativo = nota de crédito o ajuste (`is_adjustment`), se conserva.
**Facturas en USD con tipo de cambio ≠ 1 (D23):** se usa 1 y queda el flag `is_fx_overridden`. La regla `S04` manda a quarantine monedas desconocidas o tipos de cambio ≤ 0.

In [14]:
from src.jobs.silver import build_silver_billing, build_silver_tickets

with log.track("silver_billing_usd", "silver", "billing_usd"):
    billing = build_silver_billing(spark, cfg, dim_org, RUN_ID, log)
(billing.groupBy("currency")
    .agg(F.count("*").alias("facturas"), F.round(F.min("exchange_rate_to_usd"), 5).alias("fx_min"),
         F.round(F.max("exchange_rate_to_usd"), 5).alias("fx_max"), F.round(F.avg("fx_applied"), 5).alias("fx_aplicado"),
         F.sum(F.col("is_fx_overridden").cast("int")).alias("fx_forzado_a_1"),
         F.sum(F.col("is_adjustment").cast("int")).alias("ajustes"),
         F.round(F.sum("revenue_usd"), 2).alias("revenue_usd"))
    .orderBy("currency").show())

# Impacto de D23: cuánto cambiaría el revenue USD si se aplicara el tipo de cambio que trae el archivo
fx_archivo = billing.filter("currency = 'USD'").agg(
    F.round(F.sum((F.col("subtotal") - F.col("credits") + F.col("taxes")) * F.col("exchange_rate_to_usd")), 2).alias("con_fx_archivo"),
    F.round(F.sum("revenue_usd"), 2).alias("con_fx_1")).collect()[0]
print(f"Facturas en USD → revenue con fx=1: {fx_archivo['con_fx_1']:,} | con el fx del archivo: {fx_archivo['con_fx_archivo']:,}")
billing.select("invoice_id", "org_id", "billing_month", "currency", "subtotal", "credits", "taxes",
               "fx_applied", "revenue_usd").orderBy("currency", "invoice_id").show(6)

+--------+--------+-------+-------+-----------+--------------+-------+-----------+
|currency|facturas| fx_min| fx_max|fx_aplicado|fx_forzado_a_1|ajustes|revenue_usd|
+--------+--------+-------+-------+-----------+--------------+-------+-----------+
|     ARS|      51|0.00133|0.00162|     0.0015|             0|      2|      75.74|
|     EUR|      29| 0.9981|1.19808|    1.10165|             0|      1|    31965.9|
|     USD|     160|0.85463|1.11791|        1.0|           160|     10|  132251.42|
+--------+--------+-------+-------+-----------+--------------+-------+-----------+

Facturas en USD → revenue con fx=1: 132,251.42 | con el fx del archivo: 132,143.26


+------------+------------+-------------+--------+--------+-------+------+----------+-----------+
|  invoice_id|      org_id|billing_month|currency|subtotal|credits| taxes|fx_applied|revenue_usd|
+------------+------------+-------------+--------+--------+-------+------+----------+-----------+
|inv_03ivwack|org_hv3a3zmf|      2025-06|     ARS|  350.32|    0.0| 73.57|   0.00158|     0.6697|
|inv_07uj4fqk|org_1n13jcat|      2025-06|     ARS| 1079.91|    2.3|226.78|   0.00153|     1.9957|
|inv_0kg6umh4|org_ktakpuxq|      2025-08|     ARS|-1355.74|    0.0|284.71|   0.00157|    -1.6815|
|inv_27nls4sz|org_pbhsahxt|      2025-06|     ARS|  520.91|    0.0|109.39|   0.00147|     0.9265|
|inv_2fm72c6z|org_id25owf7|      2025-08|     ARS| 1579.19|    0.0|331.63|   0.00148|      2.828|
|inv_2mwm3728|org_t9nt3w5u|      2025-07|     ARS|  345.44|  11.43| 72.54|   0.00151|     0.6139|
+------------+------------+-------------+--------+--------+-------+------+----------+-----------+
only showing top 6 r

### 4.4 Tickets de soporte (`support_tickets`)

Fechas tipadas, `sla_breached` booleano, `is_open` (sin `resolved_at`), días de resolución e `is_critical`. **CSAT (D24):** la escala esperada es 1–5 y en los datos aparecen 0, 6 y 7; los valores fuera de rango pasan a `null` con el flag `is_csat_out_of_range` para no distorsionar el promedio.

In [15]:
with log.track("silver_support_tickets", "silver", "support_tickets"):
    tickets = build_silver_tickets(spark, cfg, dim_org, RUN_ID, log)
log.flush()
tickets.groupBy("severity").agg(F.count("*").alias("tickets"),
        F.round(F.avg(F.col("sla_breached").cast("double")), 3).alias("tasa_sla_breach"),
        F.round(F.avg("csat"), 2).alias("csat_prom"),
        F.sum(F.col("is_open").cast("int")).alias("abiertos")).orderBy("severity").show()
print("CSAT fuera de [1, 5] →", tickets.filter("is_csat_out_of_range").count(),
      "| rango de fechas:", tickets.agg(F.min("created_date"), F.max("created_date")).collect()[0])

+--------+-------+---------------+---------+--------+
|severity|tickets|tasa_sla_breach|csat_prom|abiertos|
+--------+-------+---------------+---------+--------+
|critical|     56|          0.036|     3.25|      18|
|    high|    204|          0.098|     3.17|      50|
|     low|    412|          0.087|     3.36|      92|
|  medium|    328|          0.113|     3.13|      80|
+--------+-------+---------------+---------+--------+

CSAT fuera de [1, 5] → 40 | rango de fechas: Row(min(created_date)=datetime.date(2025, 5, 9), max(created_date)=datetime.date(2025, 8, 31))


## 5. Componente analítico: anomalías de costo

No hay una variable objetivo etiquetada (nadie marcó qué costos son anómalos), así que en lugar de un modelo supervisado de Spark ML usamos un método **no supervisado y robusto** de los que menciona la consigna (z-score, MAD o percentiles), tal como quedó planteado en la E1 (D09):

`mad_score = 0.6745 · |costo − mediana| / MAD`, calculado **por (servicio, métrica)**, y `is_cost_anomaly = mad_score > 3.5`.

En la E1 vimos que con un umbral global el z-score marca 70 eventos y el MAD el 25 %: el costo depende del servicio y la métrica. Los parámetros de cada grupo quedan guardados en `_meta/anomaly_params` para poder auditar por qué un evento fue marcado.

In [16]:
params = spark.read.parquet(f"{P['meta']}/anomaly_params")
anom = (silver_ev.groupBy("service", "metric")
        .agg(F.count("*").alias("eventos"),
             F.sum(F.col("is_cost_anomaly").cast("int")).alias("anomalias"))
        .join(params.select("service", "metric", F.round("cost_median", 3).alias("mediana"),
                            F.round("cost_mad", 3).alias("mad")), ["service", "metric"])
        .withColumn("tasa_%", F.round(100 * F.col("anomalias") / F.col("eventos"), 2))
        .orderBy("service", "metric"))
anom.show(20)
anom_pd = anom.toPandas()

tot = silver_ev.agg(F.count("*").alias("n"), F.sum(F.col("is_cost_anomaly").cast("int")).alias("a")).collect()[0]
tasa = 100 * tot["a"] / tot["n"]
print(f"Anomalías: {tot['a']:,} de {tot['n']:,} eventos → {tasa:.2f} % (objetivo O7: revisar si supera ~1 %)")

+----------+----------------+-------+---------+-------+-----+------+
|   service|          metric|eventos|anomalias|mediana|  mad|tasa_%|
+----------+----------------+-------+---------+-------+-----+------+
| analytics|       cpu_hours|   1148|        6|  0.128|0.071|  0.52|
| analytics|        requests|   2075|       10|  8.362|0.954|  0.48|
| analytics|storage_gb_hours|   1367|        5|  0.708|0.381|  0.37|
|   compute|       cpu_hours|   3143|       10|  0.141|0.083|  0.32|
|   compute|        requests|   5568|       29|  9.535|1.173|  0.52|
|   compute|storage_gb_hours|   3785|       24|  0.772|0.431|  0.63|
|  database|       cpu_hours|   1767|        7|  0.093| 0.05|   0.4|
|  database|        requests|   3357|       14|  5.965|0.708|  0.42|
|  database|storage_gb_hours|   2294|        9|  0.502|0.285|  0.39|
|     genai|       cpu_hours|   1076|        3|  0.207|0.124|  0.28|
|     genai|        requests|   1844|        6| 14.456|1.657|  0.33|
|     genai|storage_gb_hours|   12

Anomalías: 191 de 43,196 eventos → 0.44 % (objetivo O7: revisar si supera ~1 %)


In [17]:
# Contexto de las anomalías más fuertes (lo que vería un analista de FinOps)
(silver_ev.filter("is_cost_anomaly")
    .select("event_date", "org_name", "service", "metric", "value_num", "cost_usd_increment", "mad_score")
    .orderBy(F.desc("mad_score")).show(8, truncate=False))

# Comparación con el z-score global (el otro método evaluado en la E1) sobre los mismos datos de Silver
st = silver_ev.agg(F.mean("cost_usd_increment").alias("m"), F.stddev("cost_usd_increment").alias("s")).collect()[0]
comp = (silver_ev.withColumn("z_global_3", (F.abs((F.col("cost_usd_increment") - st["m"]) / st["s"]) > 3).cast("int"))
        .groupBy("service")
        .agg(F.sum("z_global_3").alias("z_score_global>3"),
             F.sum(F.col("is_cost_anomaly").cast("int")).alias("mad_por_grupo>3.5"))
        .orderBy("service"))
comp.show()

+----------+----------------+----------+--------+---------+------------------+---------+
|event_date|org_name        |service   |metric  |value_num|cost_usd_increment|mad_score|
+----------+----------------+----------+--------+---------+------------------+---------+
|2025-08-25|Nimbus Data 18  |analytics |requests|131.0    |249.3861          |170.463  |
|2025-07-09|Apex Cloud 34   |compute   |requests|133.0    |302.3459          |168.416  |
|2025-08-29|Nova Cloud 45   |analytics |requests|128.0    |221.2907          |150.593  |
|2025-07-09|Omega Digital 40|compute   |requests|151.0    |261.3299          |144.825  |
|2025-08-20|Delta Digital 10|storage   |requests|124.0    |62.9922           |142.946  |
|2025-07-21|Delta Tech 52   |analytics |requests|127.0    |209.8516          |142.503  |
|2025-07-23|Alpha Labs 21   |database  |requests|125.0    |149.9091          |137.191  |
|2025-07-11|Gamma Labs 73   |networking|requests|125.0    |29.6319           |132.787  |
+----------+---------

+----------+----------------+-----------------+
|   service|z_score_global>3|mad_por_grupo>3.5|
+----------+----------------+-----------------+
| analytics|              10|               21|
|   compute|              30|               63|
|  database|              11|               30|
|     genai|               5|               15|
|networking|               1|               27|
|   storage|              13|               35|
+----------+----------------+-----------------+



**Lectura de resultados.** Se marcan 191 eventos (0,44 %), por debajo del ~1 % que nos pusimos como techo en O7, y las alertas se reparten entre todos los servicios (entre 0,27 % y 0,69 % por grupo). El z-score global, en cambio, depende de la escala del costo: en `networking`, que es barato, marca solo 1 evento, porque un spike de 30 USD queda "normal" comparado con los costos de compute o GenAI. Las anomalías más fuertes son requests con un costo de alrededor de 30 veces la mediana de su grupo, que es justo lo que FinOps quiere revisar. **Limitaciones:** no hay etiquetas para medir precisión/recall; el umbral 3.5 es el valor estándar del método (Iglewicz y Hoaglin) y no está calibrado con feedback de negocio. El resultado se publica en el mart `cost_anomaly_mart` (sección 6.2), con el contexto de cada día: cuánto del costo viene de eventos anómalos y cómo se compara con el día típico de esa org y servicio.

## 6. Gold

### 6.1 Mart FinOps `org_daily_usage_by_service`

Grano: **(org_id, usage_date, service)**. Es la misma agregación que validamos con MapReduce en la E1, ahora desde Silver (datos limpios). Se escribe particionado por `usage_date`.
Además se precalcula `top_services_14d` (ranking de servicios por costo de los últimos 14 días para cada org y fecha), porque Cassandra no puede ordenar por un agregado (D13).

In [18]:
from src.jobs.gold import build_gold

with log.track("gold_org_daily_usage", "gold", "org_daily_usage_by_service"):
    mart, top = build_gold(spark, cfg, RUN_ID, log)
log.flush()
mart.printSchema()

dup_grano = mart.groupBy("org_id", "usage_date", "service").count().filter("count > 1").count()
cost_silver = silver_ev.agg(F.round(F.sum("cost_usd_increment"), 2)).collect()[0][0]
cost_gold = mart.agg(F.round(F.sum("daily_cost_usd"), 2)).collect()[0][0]
print(f"Filas del mart: {mart.count():,} | claves duplicadas: {dup_grano}")
print(f"Conciliación de costos → Silver: {cost_silver:,.2f} USD | Gold: {cost_gold:,.2f} USD | "
      f"diferencia: {abs(cost_silver - cost_gold):.2f} (redondeo a 4 decimales por fila)")

root
 |-- org_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- daily_cost_usd: double (nullable = true)
 |-- requests: double (nullable = true)
 |-- cpu_hours: double (nullable = true)
 |-- storage_gb_hours: double (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- events: long (nullable = true)
 |-- anomaly_events: long (nullable = true)
 |-- run_id: string (nullable = true)
 |-- usage_date: date (nullable = true)



Filas del mart: 11,050 | claves duplicadas: 0
Conciliación de costos → Silver: 149,246.81 USD | Gold: 149,246.81 USD | diferencia: 0.00 (redondeo a 4 decimales por fila)


In [19]:
(mart.orderBy(F.desc("daily_cost_usd"))
     .select("org_id", "usage_date", "service", "daily_cost_usd", "requests", "cpu_hours",
             "storage_gb_hours", "genai_tokens", "carbon_kg", "events", "anomaly_events")
     .show(5))

# Org de ejemplo para las consultas: la de mayor costo en los últimos 14 días
ultima_fecha = mart.agg(F.max("usage_date")).collect()[0][0]
ORG_EJEMPLO = (top.filter(F.col("as_of_date") == ultima_fecha).filter("rank = 1")
                  .orderBy(F.desc("cost_14d_usd")).first()["org_id"])
print("Última fecha:", ultima_fecha, "| org de ejemplo:", ORG_EJEMPLO)
top.filter((F.col("org_id") == ORG_EJEMPLO) & (F.col("as_of_date") == ultima_fecha)).orderBy("rank").show()

+------------+----------+-------+--------------+--------+---------+----------------+------------+---------+------+--------------+
|      org_id|usage_date|service|daily_cost_usd|requests|cpu_hours|storage_gb_hours|genai_tokens|carbon_kg|events|anomaly_events|
+------------+----------+-------+--------------+--------+---------+----------------+------------+---------+------+--------------+
|org_rixa11dp|2025-08-24|  genai|      340.5259|   390.0|   0.0473|          8.3086|        5646|   0.0797|     6|             1|
|org_c11ertj5|2025-07-11|  genai|      334.6767|   237.0|     NULL|         12.9349|        NULL|     NULL|     3|             1|
|org_tvhhpbmy|2025-07-04|  genai|      313.4168|   210.0|   1.3851|            NULL|        NULL|     NULL|     3|             1|
|org_x7eedtjv|2025-07-09|compute|       311.255|   249.0|   1.8107|            NULL|        NULL|     NULL|     3|             1|
|org_5935a0l7|2025-07-09|compute|      271.6404|   267.0|   1.1745|          7.2181|      

Última fecha: 2025-08-31 | org de ejemplo: org_ktakpuxq
+------------+----------+----+----------+------------+------------+--------------------+
|      org_id|as_of_date|rank|   service|cost_14d_usd|requests_14d|              run_id|
+------------+----------+----+----------+------------+------------+--------------------+
|org_ktakpuxq|2025-08-31|   1|   compute|    659.5472|      7385.0|20261118T130005-c...|
|org_ktakpuxq|2025-08-31|   2|     genai|    141.8207|      1115.0|20261118T130005-c...|
|org_ktakpuxq|2025-08-31|   3|  database|     64.3944|      1296.0|20261118T130005-c...|
|org_ktakpuxq|2025-08-31|   4|networking|      8.2631|       733.0|20261118T130005-c...|
+------------+----------+----+----------+------------+------------+--------------------+



### 6.2 Marts de FinOps, Soporte y Producto

| Dominio | Mart | Grano | Consulta |
|---|---|---|---|
| FinOps | `revenue_by_org_month` | (org_id, billing_month) | Q4 |
| FinOps | `cost_anomaly_mart` | (org_id, usage_date, service) | — (consumo analítico) |
| Soporte | `tickets_by_org_date` | (org_id, ticket_date, severity) | base de Q3 |
| Soporte | `tickets_by_severity_date` | (severity, ticket_date), con todos los días y `severity = 'all'` | Q3 |
| Producto / GenAI | `genai_tokens_by_org_date` | (org_id, usage_date) | Q5 |

`tickets_by_severity_date` es la vista de `tickets_by_org_date` agregada para Q3: la consigna pide la evolución diaria **sin filtrar por organización** (D25), y se completan los días sin tickets con 0 para que la serie no tenga huecos.

In [20]:
from src.jobs.gold import build_gold_final

with log.track("gold_final", "gold", "marts"):
    gold = build_gold_final(spark, cfg, RUN_ID, log)
log.flush()

def grano_unico(df, cols):
    return df.groupBy(*cols).count().filter("count > 1").count() == 0

granos = {"revenue_by_org_month": ["org_id", "billing_month"],
          "tickets_by_org_date": ["org_id", "ticket_date", "severity"],
          "tickets_by_severity_date": ["severity", "ticket_date"],
          "genai_tokens_by_org_date": ["org_id", "usage_date"],
          "cost_anomaly_mart": ["org_id", "usage_date", "service"]}
print(f"{'mart':<28}{'filas':>8}  grano único")
for name, cols in granos.items():
    print(f"{name:<28}{gold[name].count():>8,}  {grano_unico(gold[name], cols)}")

# Conciliaciones con Silver
rev_s = billing.agg(F.round(F.sum("revenue_usd"), 2)).collect()[0][0]
rev_g = gold["revenue_by_org_month"].agg(F.round(F.sum("revenue_usd"), 2)).collect()[0][0]
tk_g = gold["tickets_by_org_date"].agg(F.sum("tickets")).collect()[0][0]
all_g = gold["tickets_by_severity_date"].filter("severity = 'all'").agg(F.sum("tickets")).collect()[0][0]
tok_s = silver_ev.filter("service = 'genai'").agg(F.sum("genai_tokens")).collect()[0][0]
tok_g = gold["genai_tokens_by_org_date"].agg(F.sum("genai_tokens")).collect()[0][0]
print(f"\nRevenue USD  → Silver {rev_s:,.2f} | Gold {rev_g:,.2f}")
print(f"Tickets      → Silver {tickets.count():,} | Gold por org {tk_g:,} | Gold 'all' {all_g:,}")
print(f"Tokens GenAI → Silver {tok_s:,} | Gold {tok_g:,}")

mart                           filas  grano único


revenue_by_org_month             240  True
tickets_by_org_date              984  True


tickets_by_severity_date         575  True
genai_tokens_by_org_date       1,131  True


cost_anomaly_mart             11,050  True



Revenue USD  → Silver 164,293.06 | Gold 164,293.07
Tickets      → Silver 1,000 | Gold por org 1,000 | Gold 'all' 1,000
Tokens GenAI → Silver 2,558,359 | Gold 2,558,359


In [21]:
# Org de ejemplo para Q5: la de más tokens GenAI
ORG_GENAI = gold["genai_tokens_by_org_date"].groupBy("org_id").agg(F.sum("genai_tokens").alias("t")) \
    .orderBy(F.desc("t")).first()["org_id"]
ultima_fecha_tk = gold["tickets_by_severity_date"].agg(F.max("ticket_date")).collect()[0][0]
print("Org con más tokens:", ORG_GENAI, "| último día con tickets:", ultima_fecha_tk)

gold["revenue_by_org_month"].filter(F.col("org_id") == ORG_EJEMPLO).orderBy("billing_month").show()
gold["tickets_by_severity_date"].filter("severity = 'critical'").orderBy(F.desc("ticket_date")).show(5)
gold["genai_tokens_by_org_date"].filter(F.col("org_id") == ORG_GENAI).orderBy(F.desc("usage_date")).show(5)
(gold["cost_anomaly_mart"].filter("is_anomalous_day").orderBy(F.desc("anomalous_cost_usd"))
    .select("org_name", "usage_date", "service", "daily_cost_usd", "anomalous_cost_usd", "max_mad_score",
            "median_daily_cost_usd", "cost_vs_median").show(5, truncate=False))

Org con más tokens: org_pbhsahxt | último día con tickets: 2025-08-31
+------------+----------------+--------+----------+------------+-----------+---------+-----------+----------+-----------+-------------+--------------------+-------------+
|      org_id|        org_name|invoices|currencies|subtotal_usd|credits_usd|taxes_usd|revenue_usd|fx_applied|adjustments|fx_overridden|              run_id|billing_month|
+------------+----------------+--------+----------+------------+-----------+---------+-----------+----------+-----------+-------------+--------------------+-------------+
|org_ktakpuxq|Delta Digital 49|       1|       USD|      283.18|        0.0|    59.47|     342.65|       1.0|          0|            1|20261118T130005-c...|      2025-06|
|org_ktakpuxq|Delta Digital 49|       1|       USD|      654.61|        0.0|   137.47|     792.08|       1.0|          0|            1|20261118T130005-c...|      2025-07|
|org_ktakpuxq|Delta Digital 49|       1|       ARS|       -2.13|        0.0

+--------+-----------+-------+------------+--------+---------------+--------------------+
|severity|ticket_date|tickets|sla_breached|csat_avg|sla_breach_rate|              run_id|
+--------+-----------+-------+------------+--------+---------------+--------------------+
|critical| 2025-08-31|      1|           0|    NULL|            0.0|20261118T130005-c...|
|critical| 2025-08-30|      0|           0|    NULL|           NULL|20261118T130005-c...|
|critical| 2025-08-29|      0|           0|    NULL|           NULL|20261118T130005-c...|
|critical| 2025-08-28|      1|           0|     3.0|            0.0|20261118T130005-c...|
|critical| 2025-08-27|      1|           0|     4.0|            0.0|20261118T130005-c...|
+--------+-----------+-------+------------+--------+---------------+--------------------+
only showing top 5 rows

+------------+----------+-----------+------------+--------------+--------+------+----------------------+--------------------+
|      org_id|usage_date|   org_name|ge

+----------------+----------+-------+--------------+------------------+-------------+---------------------+--------------+
|org_name        |usage_date|service|daily_cost_usd|anomalous_cost_usd|max_mad_score|median_daily_cost_usd|cost_vs_median|
+----------------+----------+-------+--------------+------------------+-------------+---------------------+--------------+
|Apex Cloud 17   |2025-07-11|genai  |334.6767      |317.4308          |123.329      |18.3349              |18.25         |
|Vertex AI 53    |2025-08-24|genai  |340.5259      |307.0602          |119.108      |27.8162              |12.24         |
|Apex Cloud 34   |2025-07-09|compute|311.255       |302.3459          |168.416      |12.9288              |24.07         |
|Nova Tech 61    |2025-07-04|genai  |313.4168      |300.396           |116.395      |18.4484              |16.99         |
|Omega Digital 40|2025-07-09|compute|271.6404      |261.3299          |144.825      |19.9286              |13.63         |
+---------------

## 7. Idempotencia

1. **Streaming:** se vuelve a lanzar la misma query; el checkpoint ya registra los 120 archivos como procesados, así que no lee nada nuevo.
2. **Batch (Bronze, Silver, Gold):** se re-ejecutan los jobs; el dedupe por clave natural + overwrite dinámico por partición dejan los mismos conteos.

In [22]:
def conteos():
    return {
        "bronze.usage_events": spark.read.parquet(f"{P['bronze']}/usage_events").count(),
        "bronze.customers_orgs": spark.read.parquet(f"{P['bronze']}/customers_orgs").count(),
        "bronze.users": spark.read.parquet(f"{P['bronze']}/users").count(),
        "bronze.billing_monthly": spark.read.parquet(f"{P['bronze']}/billing_monthly").count(),
        "silver.usage_events": spark.read.parquet(f"{P['silver']}/usage_events").count(),
        "silver.dim_org": spark.read.parquet(f"{P['silver']}/dim_org").count(),
        "quarantine.usage_events_silver": spark.read.parquet(f"{P['quarantine']}/usage_events_silver").count(),
        "gold.org_daily_usage_by_service": spark.read.parquet(f"{P['gold']}/org_daily_usage_by_service").count(),
        "gold.top_services_14d": spark.read.parquet(f"{P['gold']}/top_services_14d").count(),
        "silver.billing_usd": spark.read.parquet(f"{P['silver']}/billing_usd").count(),
        "silver.support_tickets": spark.read.parquet(f"{P['silver']}/support_tickets").count(),
        **{f"gold.{n}": spark.read.parquet(f"{P['gold']}/{n}").count() for n in granos},
    }

antes = conteos()

# 1) streaming: misma query, mismo checkpoint
q1, q2 = start_bronze_queries(spark, cfg, RUN_ID)
q1.awaitTermination(); q2.awaitTermination()
print("Re-ejecución del stream →", progress_summary(q1))

# 2) batch completo otra vez
log_re = RunLog(spark, P["meta"], RUN_ID + "-reejecucion")
for name in CSV_SCHEMAS:
    ingest_master(spark, cfg, name, log_re.run_id, log_re)
dim_org = build_dim_org(spark, cfg, log_re.run_id, log_re)
build_silver_events(spark, cfg, dim_org, log_re.run_id, log_re)
build_silver_billing(spark, cfg, dim_org, log_re.run_id, log_re)
build_silver_tickets(spark, cfg, dim_org, log_re.run_id, log_re)
build_gold(spark, cfg, log_re.run_id, log_re)
build_gold_final(spark, cfg, log_re.run_id, log_re)
log_re.flush()

despues = conteos()
idem = [(k, antes[k], despues[k], antes[k] == despues[k]) for k in antes]
print(f"\n{'dataset':<34}{'antes':>9}{'después':>9}  igual")
for k, a, d, ok in idem:
    print(f"{k:<34}{a:>9,}{d:>9,}  {ok}")
print("\nTodo idéntico:", all(x[3] for x in idem))

# los DataFrames de las secciones anteriores apuntan a archivos que la re-ejecución reemplazó: se releen
silver_ev = spark.read.parquet(f"{P['silver']}/usage_events")
quar_silver = spark.read.parquet(f"{P['quarantine']}/usage_events_silver")
mart = spark.read.parquet(f"{P['gold']}/org_daily_usage_by_service")
top = spark.read.parquet(f"{P['gold']}/top_services_14d")
billing = spark.read.parquet(f"{P['silver']}/billing_usd")
tickets = spark.read.parquet(f"{P['silver']}/support_tickets")
gold = {n: spark.read.parquet(f"{P['gold']}/{n}") for n in granos}

Re-ejecución del stream → {'micro_lotes': 0, 'filas_leidas': 0, 'descartadas_por_watermark': 0, 'emitidas': 0, 'claves_en_estado': 0}



dataset                               antes  después  igual
bronze.usage_events                  43,200   43,200  True
bronze.customers_orgs                    80       80  True
bronze.users                            800      800  True
bronze.billing_monthly                  240      240  True
silver.usage_events                  43,196   43,196  True
silver.dim_org                           80       80  True
quarantine.usage_events_silver            4        4  True
gold.org_daily_usage_by_service      11,050   11,050  True
gold.top_services_14d                12,314   12,314  True
silver.billing_usd                      240      240  True
silver.support_tickets                1,000    1,000  True
gold.revenue_by_org_month               240      240  True
gold.tickets_by_org_date                984      984  True
gold.tickets_by_severity_date           575      575  True
gold.genai_tokens_by_org_date         1,131    1,131  True
gold.cost_anomaly_mart               11,050   11,050  

## 8. Serving en Cassandra / AstraDB

Modelo **query-first** (`cql/01_schema.cql`): una tabla por consulta.

| Consulta | Tabla | Partition key | Clustering | Por qué |
|---|---|---|---|---|
| Q1 costos y requests diarios por org y servicio en un rango | `org_daily_usage_by_service` | `org_id` | `usage_date DESC, service` | El rango de fechas se resuelve dentro de una sola partición |
| Q2 Top-N servicios por costo de los últimos 14 días | `top_services_14d` | `(org_id, as_of_date)` | `rank ASC` | Cassandra no ordena por un agregado: el ranking viene precalculado y `LIMIT N` alcanza |
| Q3 tickets críticos y tasa de SLA breach por día, últimos 30 días | `tickets_by_severity_date` | `severity` | `ticket_date DESC` | 1 partición por severidad con una fila por día; 30 días = las primeras 30 filas |
| Q4 revenue mensual con créditos e impuestos, en USD | `revenue_by_org_month` | `org_id` | `billing_month DESC` | La serie mensual de una org en una partición |
| Q5 tokens GenAI y costo estimado por día | `genai_tokens_by_org_date` | `org_id` | `usage_date DESC` | El rango de fechas se lee de una sola partición |

La carga usa el driver de Python con sentencias preparadas y escritura concurrente. Como `INSERT` en Cassandra es un **upsert por PRIMARY KEY**, recargar no duplica. Las sentencias usan nombres calificados (`cloud_analytics.tabla`) y protocolo v4, como recomienda el driver para AstraDB.

Credenciales: el token se lee de los *Secrets* de Colab (`ASTRA_DB_APPLICATION_TOKEN`) y el bundle de `/content/secure-connect-big-data.zip`. Nada de esto está en el repo.

In [23]:
SCB_PATH = f"{P['base']}/secure-connect-big-data.zip"
try:
    from google.colab import userdata
    TOKEN = userdata.get(cfg["cassandra"]["token_env"])
except Exception:
    TOKEN = os.environ.get(cfg["cassandra"]["token_env"])

HAY_ASTRA = bool(TOKEN) and os.path.exists(SCB_PATH)
print("Credenciales de AstraDB disponibles:", HAY_ASTRA)
if not HAY_ASTRA:
    print("Sin token o sin bundle: se saltea la sección 8 (el resto del pipeline no depende de ella).")

Credenciales de AstraDB disponibles: False
Sin token o sin bundle: se saltea la sección 8 (el resto del pipeline no depende de ella).


In [24]:
if HAY_ASTRA:
    from src.jobs.serving import connect, run_cql_file, load_dataframe, count_rows
    KS = cfg["cassandra"]["keyspace"]
    cluster, session = connect(SCB_PATH, TOKEN)
    print("Conectado a", session.execute("SELECT release_version FROM system.local").one()[0], "| keyspace:", KS)
    print("Sentencias ejecutadas de cql/01_schema.cql:", run_cql_file(session, f"{REPO}/cql/01_schema.cql", KS))

In [25]:
if HAY_ASTRA:
    MART_CQL = ["org_id", "usage_date", "service", "org_name", "daily_cost_usd", "requests", "cpu_hours",
                "storage_gb_hours", "genai_tokens", "carbon_kg", "events", "anomaly_events", "run_id"]
    TOP_CQL = ["org_id", "as_of_date", "rank", "service", "cost_14d_usd", "requests_14d", "run_id"]
    mart_c = (mart.withColumn("events", F.col("events").cast("int"))
                  .withColumn("anomaly_events", F.col("anomaly_events").cast("int")))
    orgs = [r[0] for r in mart.select("org_id").distinct().collect()]

    CARGAS = {
        "org_daily_usage_by_service": (mart_c, MART_CQL),
        "top_services_14d": (top, TOP_CQL),
        "tickets_by_severity_date": (gold["tickets_by_severity_date"],
            ["severity", "ticket_date", "tickets", "sla_breached", "sla_breach_rate", "csat_avg", "run_id"]),
        "revenue_by_org_month": (gold["revenue_by_org_month"],
            ["org_id", "billing_month", "org_name", "invoices", "currencies", "subtotal_usd", "credits_usd",
             "taxes_usd", "revenue_usd", "fx_applied", "adjustments", "run_id"]),
        "genai_tokens_by_org_date": (gold["genai_tokens_by_org_date"],
            ["org_id", "usage_date", "org_name", "genai_tokens", "genai_cost_usd", "cost_per_1k_tokens_usd",
             "requests", "events", "run_id"]),
    }
    for tabla, (df, cols) in CARGAS.items():
        t0 = time.time()
        enviadas, errores = load_dataframe(session, df, f"{KS}.{tabla}", cols, cfg["cassandra"]["concurrency"])
        print(f"{tabla:<28} → enviadas: {enviadas:>6,} | errores: {errores} | {time.time()-t0:.0f} s")
        log.add("serving_cassandra", "serving", tabla, enviadas, enviadas - errores, conciliar=False,
                detalle=f"errores={errores}")

    filas_cassandra = count_rows(session, f"{KS}.org_daily_usage_by_service", orgs)
    print(f"\nFilas en Cassandra (org_daily_usage_by_service): {filas_cassandra:,} | en Gold: {mart.count():,}")
    log.flush()

In [26]:
if HAY_ASTRA:
    import pandas as pd
    q1 = f'''SELECT usage_date, service, daily_cost_usd, requests, events, anomaly_events
FROM {KS}.org_daily_usage_by_service
WHERE org_id = '{ORG_EJEMPLO}'
  AND usage_date >= '2025-08-18' AND usage_date <= '2025-08-31';'''
    print("Q1 · CQL:\n" + q1)
    res_q1 = pd.DataFrame(list(session.execute(q1.rstrip(";"))))
    print(f"\n{len(res_q1)} filas")
    display(res_q1.head(15))

In [27]:
if HAY_ASTRA:
    q2 = f'''SELECT rank, service, cost_14d_usd, requests_14d
FROM {KS}.top_services_14d
WHERE org_id = '{ORG_EJEMPLO}' AND as_of_date = '{ultima_fecha}'
LIMIT 3;'''
    print("Q2 · CQL:\n" + q2)
    res_q2 = pd.DataFrame(list(session.execute(q2.rstrip(";"))))
    display(res_q2)

    # Verificación cruzada: lo que devuelve Cassandra coincide con Gold
    esperado = (top.filter((F.col("org_id") == ORG_EJEMPLO) & (F.col("as_of_date") == ultima_fecha))
                   .orderBy("rank").limit(3).select("service").toPandas()["service"].tolist())
    print("Coincide con Gold:", esperado == res_q2["service"].tolist())

In [28]:
if HAY_ASTRA:
    import datetime as dt
    desde_q3 = ultima_fecha_tk - dt.timedelta(days=29)
    q3 = f'''SELECT ticket_date, tickets, sla_breached, sla_breach_rate
FROM {KS}.tickets_by_severity_date
WHERE severity = 'critical' AND ticket_date >= '{desde_q3}' AND ticket_date <= '{ultima_fecha_tk}';'''
    print("Q3 · CQL:\n" + q3)
    res_q3 = pd.DataFrame(list(session.execute(q3.rstrip(";"))))
    print(f"\n{len(res_q3)} días | tickets críticos: {int(res_q3['tickets'].sum())} | "
          f"con SLA breach: {int(res_q3['sla_breached'].sum())}")
    display(res_q3.head(10))

In [29]:
if HAY_ASTRA:
    q4 = f'''SELECT billing_month, currencies, subtotal_usd, credits_usd, taxes_usd, revenue_usd, fx_applied
FROM {KS}.revenue_by_org_month
WHERE org_id = '{ORG_EJEMPLO}';'''
    print("Q4 · CQL:\n" + q4)
    res_q4 = pd.DataFrame(list(session.execute(q4.rstrip(";"))))
    display(res_q4)

In [30]:
if HAY_ASTRA:
    q5 = f'''SELECT usage_date, genai_tokens, genai_cost_usd, cost_per_1k_tokens_usd
FROM {KS}.genai_tokens_by_org_date
WHERE org_id = '{ORG_GENAI}' AND usage_date >= '2025-08-18' AND usage_date <= '2025-08-31';'''
    print("Q5 · CQL:\n" + q5)
    res_q5 = pd.DataFrame(list(session.execute(q5.rstrip(";"))))
    print(f"\n{len(res_q5)} días | tokens: {int(res_q5['genai_tokens'].sum()):,}")
    display(res_q5)

In [31]:
if HAY_ASTRA:
    # Idempotencia en serving: recargar el mismo mart no cambia la cantidad de filas (upsert por PK)
    antes_c = count_rows(session, f"{KS}.org_daily_usage_by_service", orgs)
    load_dataframe(session, mart_c, f"{KS}.org_daily_usage_by_service", MART_CQL, cfg["cassandra"]["concurrency"])
    despues_c = count_rows(session, f"{KS}.org_daily_usage_by_service", orgs)
    print(f"Cassandra antes: {antes_c:,} | después de recargar: {despues_c:,} | idéntico: {antes_c == despues_c}")
    for i, r in enumerate([res_q1, res_q2, res_q3, res_q4, res_q5], start=1):
        r.to_csv(f"{P['evidence']}/final_cassandra_q{i}.csv", index=False)
    cluster.shutdown()

## 9. Evidencias: run_log, rutas, tamaños y quarantine

In [32]:
run_log = log.read()
(run_log.filter(F.col("run_id") == RUN_ID)
    .select("job", "dataset", "filas_entrada", "filas_validas", "filas_quarantine", "filas_duplicadas",
            "duracion_s", "estado", "detalle")
    .orderBy("ts").show(30, truncate=60))
print("Ejecuciones con conciliación fallida:", run_log.filter("estado <> 'OK'").count())

+-----------------------------+--------------------------+-------------+-------------+----------------+----------------+----------+------+-----------------------------------------------------------+
|                          job|                   dataset|filas_entrada|filas_validas|filas_quarantine|filas_duplicadas|duracion_s|estado|                                                    detalle|
+-----------------------------+--------------------------+-------------+-------------+----------------+----------------+----------+------+-----------------------------------------------------------+
|                 bronze_batch|            customers_orgs|           80|           80|               0|               0|      6.89|    OK|                                      particion=ingest_date|
|                 bronze_batch|                     users|          800|          800|               0|               0|       1.6|    OK|                                      particion=ingest_date|
|    

In [33]:
def zona_stats(zona_path):
    filas = []
    for ds in sorted(os.listdir(zona_path)):
        files = glob.glob(f"{zona_path}/{ds}/**/*.parquet", recursive=True)
        parts = {os.path.dirname(f) for f in files}
        filas.append((os.path.basename(zona_path), ds, len(parts), len(files),
                      round(sum(os.path.getsize(f) for f in files) / 1024, 1)))
    return filas

stats = []
for z in ["bronze", "silver", "gold", "quarantine"]:
    stats += zona_stats(P[z])
print(f"{'zona':<12}{'dataset':<32}{'particiones':>12}{'archivos':>10}{'KB':>10}")
for z, ds, p, f, kb in stats:
    print(f"{z:<12}{ds:<32}{p:>12}{f:>10}{kb:>10}")
print("\nEjemplos de rutas:")
for x in sorted(glob.glob(f"{P['gold']}/org_daily_usage_by_service/*"))[:2] + \
         sorted(glob.glob(f"{P['quarantine']}/usage_events_silver/*")):
    print("  ", x)

zona        dataset                          particiones  archivos        KB
bronze      billing_monthly                            3         3      22.3
bronze      customers_orgs                             1         1       7.2
bronze      marketing_touches                          1         1      23.3
bronze      nps_surveys                                1         1       4.2
bronze      resources                                  1         1      12.1
bronze      support_tickets                            1         1      19.3
bronze      usage_events                              60       240    4027.2
bronze      users                                      1         1      24.0
silver      billing_usd                                3         3      37.5
silver      dim_org                                    1         1       7.4
silver      support_tickets                            1         1      22.8
silver      usage_events                              60        60    3535.2

In [34]:
import pandas as pd
from pyspark.sql import Window
ev = P["evidence"]
run_log.orderBy("ts").toPandas().to_csv(f"{ev}/final_run_log.csv", index=False)
pd.DataFrame(stats, columns=["zona", "dataset", "particiones", "archivos", "kb"]).to_csv(f"{ev}/final_lake_stats.csv", index=False)
pd.DataFrame(experimento).to_csv(f"{ev}/final_watermark_experimento.csv", index=False)
(quar_silver.withColumn("rn", F.row_number().over(Window.partitionBy("rule_id").orderBy("event_id")))
    .filter("rn <= 5").drop("rn").toPandas().to_csv(f"{ev}/final_quarantine_muestras.csv", index=False))
anom_pd.to_csv(f"{ev}/final_anomalias_por_grupo.csv", index=False)
mart.orderBy("org_id", "usage_date", "service").limit(200).toPandas().to_csv(f"{ev}/final_gold_muestra.csv", index=False)
for n in granos:
    gold[n].limit(200).toPandas().to_csv(f"{ev}/final_gold_{n}_muestra.csv", index=False)

resumen = {
    "run_id": RUN_ID, "spark_version": spark.version,
    "streaming": {"bronze": prog_bronze, "watermark": cfg["streaming"]["watermark"],
                  "max_files_per_trigger": cfg["streaming"]["max_files_per_trigger"]},
    "conteos": despues,
    "calidad": {r["rule_id"]: r["count"] for r in quar_silver.groupBy("rule_id").count().collect()},
    "unit_imputadas": silver_ev.filter("unit_imputed").count(),
    "costos_signo_corregido": {"eventos": silver_ev.filter("is_cost_sign_corrected").count(),
                               "costo_original": silver_ev.filter("is_cost_sign_corrected").agg(F.round(F.sum("cost_usd_raw"), 2)).first()[0],
                               "costo_corregido": silver_ev.filter("is_cost_sign_corrected").agg(F.round(F.sum("cost_usd_increment"), 2)).first()[0]},
    "anomalias": {"eventos": tot["a"], "tasa_pct": round(tasa, 2), "metodo": "MAD por (service, metric)",
                  "umbral": cfg["quality"]["anomaly_threshold"]},
    "gold": {"org_daily_usage_by_service": {"filas": mart.count(), "claves_duplicadas": dup_grano,
                                            "costo_silver": cost_silver, "costo_gold": cost_gold},
             **{n: gold[n].count() for n in granos},
             "revenue_usd_silver": rev_s, "revenue_usd_gold": rev_g},
    "facturacion": {"facturas": billing.count(), "fx_usd_forzado_a_1": billing.filter("is_fx_overridden").count(),
                    "ajustes": billing.filter("is_adjustment").count()},
    "tickets": {"tickets": tickets.count(), "csat_fuera_de_rango": tickets.filter("is_csat_out_of_range").count()},
    "idempotencia": {k: [a, d] for k, a, d, _ in idem},
    "org_ejemplo": ORG_EJEMPLO, "org_genai": ORG_GENAI, "ultima_fecha": str(ultima_fecha),
    "ultima_fecha_tickets": str(ultima_fecha_tk),
}
with open(f"{ev}/final_resumen.json", "w") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False, default=str)
print(sorted(os.listdir(ev)))

['final_anomalias_por_grupo.csv', 'final_gold_cost_anomaly_mart_muestra.csv', 'final_gold_genai_tokens_by_org_date_muestra.csv', 'final_gold_muestra.csv', 'final_gold_revenue_by_org_month_muestra.csv', 'final_gold_tickets_by_org_date_muestra.csv', 'final_gold_tickets_by_severity_date_muestra.csv', 'final_lake_stats.csv', 'final_quarantine_muestras.csv', 'final_resumen.json', 'final_run_log.csv', 'final_watermark_experimento.csv']


### Limpieza / reinicio

Para volver a correr desde cero (por ejemplo antes de una demo) descomentar y ejecutar: borra todo el lake **menos Landing**, incluidos los checkpoints.

In [35]:
# for z in ["bronze", "silver", "gold", "quarantine", "checkpoints", "meta"]:
#     shutil.rmtree(P[z], ignore_errors=True)
# spark.stop()